# CryoSat-2 vs Jason-2 cross-over analysis: worked example

This notebook walks through the analysis from the MSc dissertation using the output of `src/xover_finder.py`:

1. Load the cross-over CSV
2. Summary statistics (mean, standard deviation)
3. Histogram and latitude scatter of ΔSLA (thesis Fig. 28)
4. Global map of ΔSLA (thesis Fig. 29)
5. ΔSLA averaged per degree of latitude (thesis Fig. 31)
6. Regression of CryoSat-2 on Jason-2 SLA: slope, intercept, *r*, *S* (thesis Fig. 36, Table 11)

**Data:** if no real cross-over file is found, the notebook generates a **synthetic** one so every cell runs. Synthetic numbers are random and have no scientific meaning. To use real data, run `python src/xover_finder.py data/ -o xover_finder.csv` from the repository root and set `CSV` below.

In [ ]:
import os, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "scripts"))
from plot_global_map import plot_map

CSV = os.path.join(ROOT, "xover_finder.csv")      # <- real output of xover_finder.py
if not os.path.exists(CSV):
    CSV = os.path.join(ROOT, "synthetic_xovers.csv")
    subprocess.run([sys.executable, os.path.join(ROOT, "scripts", "make_synthetic_data.py"), CSV], check=True)
    print("USING SYNTHETIC DATA")

d = np.genfromtxt(CSV, delimiter=",", names=True)
lat, lon = d["lat"], d["lon"]
sla_c, sla_j, dsla = d["sla_cryosat"], d["sla_jason"], d["dsla"]
print(f"{dsla.size} cross-overs")

## 1. Summary statistics

In [ ]:
print(f"CryoSat-2 SLA : mean {sla_c.mean()*100:6.2f} cm, std {sla_c.std()*100:5.2f} cm")
print(f"Jason-2 SLA   : mean {sla_j.mean()*100:6.2f} cm, std {sla_j.std()*100:5.2f} cm")
print(f"dSLA (C2-J2)  : mean {dsla.mean()*100:6.2f} cm, std {dsla.std()*100:5.2f} cm")

The **mean** of ΔSLA is the relative bias between the missions; the **standard deviation** is the cross-over noise level, the headline figure of the dissertation (6.5 cm averaged over four cycles).

## 2. Histogram and latitude scatter of ΔSLA

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7, 7), gridspec_kw={"height_ratios": [1, 2]})
ax1.hist(dsla, bins=100, color="tab:blue")
ax1.set_ylabel("Frequency")
ax1.set_title("SLA cross-over difference")
ax1.text(0.02, 0.75, f"Mean = {dsla.mean():.3f} m\nSigma = {dsla.std():.3f} m\nn = {dsla.size}",
         transform=ax1.transAxes, bbox=dict(fc="w", ec="k"))
ax2.scatter(dsla, lat, s=2, c="k")
ax2.set_xlabel("SLA cross-over difference (m)"); ax2.set_ylabel("Latitude (deg)")
plt.tight_layout(); plt.show()

The spread widening towards high latitudes reflects higher significant wave height (SIRAL height noise grows with SWH) and stronger ocean variability.

## 3. Global map of ΔSLA

In [ ]:
plot_map(lon, lat, dsla, title="SLA cross-over difference (CryoSat-2 - Jason-2)", vlim=0.5)
plt.show()

Install `cartopy` (`pip install cartopy`) to get coastlines and land shading; otherwise this is a plain lon/lat scatter.

## 4. ΔSLA averaged per degree of latitude

In [ ]:
edges = np.arange(-90, 91, 1)
centres = 0.5 * (edges[:-1] + edges[1:])
idx = np.digitize(lat, edges) - 1
mean_by_lat = np.array([dsla[idx == i].mean() if np.any(idx == i) else np.nan for i in range(len(centres))])
ok = ~np.isnan(mean_by_lat)
m, c = np.polyfit(centres[ok], mean_by_lat[ok], 1)

plt.figure(figsize=(7, 4))
plt.plot(centres[ok], mean_by_lat[ok], "k.")
plt.plot(centres[ok], m * centres[ok] + c, "r-", label=f"slope = {m:.2e} m/deg, intercept = {c:.3f} m")
plt.xlabel("Latitude (deg)"); plt.ylabel("Mean dSLA (m)"); plt.legend(); plt.grid(alpha=.3)
plt.show()

A non-zero slope would indicate a latitude-dependent (geographically correlated) bias. The dissertation found slopes of opposite sign in Northern-winter (cycles 10, 37) and summer (cycles 29, 55) data, suggesting a seasonal effect to be investigated.

## 5. Regression: CryoSat-2 SLA vs Jason-2 SLA

In [ ]:
r = np.corrcoef(sla_j, sla_c)[0, 1]
slope, intercept = np.polyfit(sla_j, sla_c, 1)
fitted = slope * sla_j + intercept
S = np.sqrt(np.sum((sla_c - fitted) ** 2) / (sla_c.size - 2))   # standard error of the regression
print(f"slope = {slope:.3f}, intercept = {intercept:.3f} m, r = {r:.3f}, S = {S*100:.2f} cm")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.plot(sla_j, sla_c, "k.", ms=1)
ax1.plot([-1, 1], [-1, 1], "k--", lw=1, label="1:1")
ax1.plot([-1, 1], [slope * -1 + intercept, slope + intercept], "r-", label="OLS fit")
ax1.set_xlim(-1, 1); ax1.set_ylim(-1, 1); ax1.set_aspect("equal")
ax1.set_xlabel("Jason-2 SLA (m)"); ax1.set_ylabel("CryoSat-2 SLA (m)"); ax1.legend()
ax2.plot(fitted, sla_c - fitted, "k.", ms=1); ax2.axhline(0, color="gray")
ax2.set_xlabel("Fitted value (m)"); ax2.set_ylabel("Residual (m)")
plt.tight_layout(); plt.show()

Residuals centred on zero with no trend against the fitted values indicate the linear model is adequate. The dissertation reports an average *r* of 0.771 and *S* of 6.1 cm across the four cycles.